In [ ]:
import pandas as pd
import numpy as np

## 1. Business case & Data Collection

## 2.Data Understanding

### 2.1 Exploratorio Inicial

In [20]:
#Cargo el dataframe desde un archivo CSV
df = pd.read_csv("./data/steel-faults.csv", sep=",")

In [21]:
#Obtengo el numero de filas y columnas de la tabla (utilizo una sola tabla)
print("Número de filas y columnas:", df.shape)

Número de filas y columnas: (1941, 34)


In [22]:
#Saco la información del dataframe (tipo de datos, cantidad de valores no nulos, etc.)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1941 entries, 0 to 1940
Data columns (total 34 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   X_Minimum              1941 non-null   int64  
 1   X_Maximum              1941 non-null   int64  
 2   Y_Minimum              1941 non-null   int64  
 3   Y_Maximum              1941 non-null   int64  
 4   Pixels_Areas           1941 non-null   int64  
 5   X_Perimeter            1941 non-null   int64  
 6   Y_Perimeter            1941 non-null   int64  
 7   Sum_of_Luminosity      1941 non-null   int64  
 8   Minimum_of_Luminosity  1941 non-null   int64  
 9   Maximum_of_Luminosity  1941 non-null   int64  
 10  Length_of_Conveyer     1941 non-null   int64  
 11  TypeOfSteel_A300       1941 non-null   int64  
 12  TypeOfSteel_A400       1941 non-null   int64  
 13  Steel_Plate_Thickness  1941 non-null   int64  
 14  Edges_Index            1941 non-null   float64
 15  Empty_Index    

In [23]:
#Muestro los primeros registros del dataframe
df.head()

,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
0,42,50,270900,270944,267,17,44,24220,76,108,...,0.8182,-0.2913,0.5822,1,0,0,0,0,0,0
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,0.7931,-0.1756,0.2984,1,0,0,0,0,0,0
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,0.6667,-0.1228,0.2150,1,0,0,0,0,0,0
3,853,860,369370,369415,176,13,45,18996,99,126,...,0.8444,-0.1568,0.5212,1,0,0,0,0,0,0
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,0.9338,-0.1992,1.0000,1,0,0,0,0,0,0


In [24]:
#Muestro los últimos registros del dataframe
df.tail()

,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Orientation_Index,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults
1936,249,277,325780,325796,273,54,22,35033,119,141,...,-0.4286,0.0026,0.7254,0,0,0,0,0,0,1
1937,144,175,340581,340598,287,44,24,34599,112,133,...,-0.4516,-0.0582,0.8173,0,0,0,0,0,0,1
1938,145,174,386779,386794,292,40,22,37572,120,140,...,-0.4828,0.0052,0.7079,0,0,0,0,0,0,1
1939,137,170,422497,422528,419,97,47,52715,117,140,...,-0.0606,-0.0171,0.9919,0,0,0,0,0,0,1
1940,1261,1281,87951,87967,103,26,22,11682,101,133,...,-0.2000,-0.1139,0.5296,0,0,0,0,0,0,1


In [25]:
#Cambio a español el nombre de las columnas que luego mostraré en los gráficos
df = df.rename(columns={
    'TypeOfSteel_A300': 'A300', 
    'TypeOfSteel_A400': 'A400', 
    'Steel_Plate_Thickness': 'Espesor',
    'Pastry': 'Pliegue',
    'Z_Scratch': 'Raya', 
    'K_Scatch': 'Surco',
    'Stains': 'Mancha',
    'Dirtiness': 'Suciedad',
    'Bumps': 'Abolladura', 
    'Other_Faults': 'Otro'
})

### 2.2 Tabla de Variables

In [26]:
#Defino las listas de vbles en fn de su naturaleza
coordenadas = ['X_Minimum', 'X_Maximum', 'Y_Minimum', 'Y_Maximum']
magnitud = ['Pixels_Areas', 'X_Perimeter', 'Y_Perimeter', 'LogOfAreas', 'Log_X_Index', 'Log_Y_Index']
intensidad = ['Sum_of_Luminosity', 'Minimum_of_Luminosity', 'Maximum_of_Luminosity', 'Luminosity_Index']
maq_mat = ['Length_of_Conveyer', 'A300', 'A400', 'Espesor']
forma = ['Edges_Index', 'Empty_Index', 'Square_Index', 'Outside_X_Index', 'Edges_X_Index', 'Edges_Y_Index', 'Outside_Global_Index', 'Orientation_Index', 'SigmoidOfAreas']
defecto = ['Pliegue', 'Raya', 'Surco', 'Mancha', 'Suciedad', 'Abolladura', 'Otro']

In [27]:
#importo las funciones del archivo funciones.py
import sys
import os
sys.path.append(os.path.abspath('../'))
from utils.funciones import asignar_naturaleza, asignar_definicion

In [28]:
#Construyo el inventario
tabla_vbles = pd.DataFrame({
    'Variable': df.columns,
    'Tipo de Dato': df.dtypes.values,
    'Naturaleza': [asignar_naturaleza(c, coordenadas, magnitud, intensidad, maq_mat, forma, defecto) for c in df.columns],
    'Descripción': [asignar_definicion(c, coordenadas, magnitud, intensidad, maq_mat, forma, defecto) for c in df.columns]
})

tabla_vbles.sort_values(by='Naturaleza').reset_index(drop=True)

,Variable,Tipo de Dato,Naturaleza,Descripción
0,A300,int64,Características de la Maquinaria y Material,Parámetros de configuración de la maquinaria y...
1,Espesor,int64,Características de la Maquinaria y Material,Parámetros de configuración de la maquinaria y...
2,A400,int64,Características de la Maquinaria y Material,Parámetros de configuración de la maquinaria y...
3,Length_of_Conveyer,int64,Características de la Maquinaria y Material,Parámetros de configuración de la maquinaria y...
4,X_Minimum,int64,Coordenadas y Ubicación Espacial (Geometría de...,"Estas variables delimitan el ""cuadro contenedo..."
5,X_Maximum,int64,Coordenadas y Ubicación Espacial (Geometría de...,"Estas variables delimitan el ""cuadro contenedo..."
6,Y_Maximum,int64,Coordenadas y Ubicación Espacial (Geometría de...,"Estas variables delimitan el ""cuadro contenedo..."
7,Y_Minimum,int64,Coordenadas y Ubicación Espacial (Geometría de...,"Estas variables delimitan el ""cuadro contenedo..."
8,Sum_of_Luminosity,int64,Intensidad y Luminosidad (Análisis Óptico),Indican cómo refleja la luz el defecto. Las fa...
9,Minimum_of_Luminosity,int64,Intensidad y Luminosidad (Análisis Óptico),Indican cómo refleja la luz el defecto. Las fa...


## 3.Data Cleaning

In [29]:
#Como ya vi en la info() del df que no hay nulos, compruebo ahora si tengo duplicados por si acaso
#importo la funcion
from utils.funciones import duplicados
duplicados(df)

El número total de elementos duplicados es de: 0


In [30]:
#Compruebo si tengo outliers
from utils.funciones import detectar_outliers
columnas_a_analizar = coordenadas + magnitud + intensidad + maq_mat + forma
tabla_outliers = detectar_outliers(df, columnas_a_analizar)
print(tabla_outliers)

                 Variable     Mínimo     Q1 (25%)  Mediana (50%)  \
0               X_Minimum     0.0000      51.0000   4.350000e+02   
1               X_Maximum     4.0000     192.0000   4.670000e+02   
2               Y_Minimum  6712.0000  471253.0000   1.204128e+06   
3               Y_Maximum  6724.0000  471281.0000   1.204136e+06   
4            Pixels_Areas     2.0000      84.0000   1.740000e+02   
5             X_Perimeter     2.0000      15.0000   2.600000e+01   
6             Y_Perimeter     1.0000      13.0000   2.500000e+01   
7              LogOfAreas     0.3010       1.9243   2.240600e+00   
8             Log_X_Index     0.3010       1.0000   1.176100e+00   
9             Log_Y_Index     0.0000       1.0792   1.322200e+00   
10      Sum_of_Luminosity   250.0000    9522.0000   1.920200e+04   
11  Minimum_of_Luminosity     0.0000      63.0000   9.000000e+01   
12  Maximum_of_Luminosity    37.0000     124.0000   1.270000e+02   
13       Luminosity_Index    -0.9989      -0.195

In [31]:
#Creo una columna categórica para el tipo de acero
from utils.funciones import asignar_tipo_acero
df['Tipo_Acero'] = df.apply(asignar_tipo_acero, axis=1)
print(df['Tipo_Acero'].value_counts())
df[['A300', 'A400', 'Tipo_Acero']].head()

Tipo_Acero
A400    1164
A300     777
Name: count, dtype: int64


,A300,A400,Tipo_Acero
0,1,0,A300
1,1,0,A300
2,1,0,A300
3,0,1,A400
4,0,1,A400


In [32]:
#creo una columna categorica para el tipo de defecto
from utils.funciones import asignar_tipo_defecto
df['Tipo_Defecto'] = df.apply(lambda fila: asignar_tipo_defecto(fila, defecto), axis=1)
print(df['Tipo_Defecto'].value_counts())
df[defecto + ['Tipo_Defecto']].head()

Tipo_Defecto
Otro          673
Abolladura    402
Surco         391
Raya          190
Pliegue       158
Mancha         72
Suciedad       55
Name: count, dtype: int64


,Pliegue,Raya,Surco,Mancha,Suciedad,Abolladura,Otro,Tipo_Defecto
0,1,0,0,0,0,0,0,Pliegue
1,1,0,0,0,0,0,0,Pliegue
2,1,0,0,0,0,0,0,Pliegue
3,1,0,0,0,0,0,0,Pliegue
4,1,0,0,0,0,0,0,Pliegue


In [33]:
#Creo una tabla Tipo de Acero vs Tipo de Defecto
tabla = pd.crosstab(df['Tipo_Acero'], df['Tipo_Defecto'], margins=True)

#Ordeno los nombres de las columnas basándonos en la fila 'All' (excluyendo el propio 'All')
columnas_ordenadas = tabla.loc['All'].drop('All').sort_values(ascending=False).index

#Reorganizo la tabla manteniendo 'All' al final
tabla[list(columnas_ordenadas) + ['All']]

Tipo_Defecto,Otro,Abolladura,Surco,Raya,Pliegue,Mancha,Suciedad,All
Tipo_Acero,,,,,,,,
A300,266,279,1,172,49,1,9,777
A400,407,123,390,18,109,71,46,1164
All,673,402,391,190,158,72,55,1941


De acuerdo a la tabla anterior:  
* El acero A300 parece vulnerable a las deformaciones físicas por impacto (Abolladura) y a las rayaaduras superficiales finas (Reyas).  
* Las rayas severas longitudinales (Surco) están ligadas de forma casi exclusiva al acero A400.
* Las manchas (Mancha) y la suciedad (Suciedad) penalizan más al A400, ya que, concentra 71 de las 72 manchas químicas (Mancha) y 46 de las 55 impurezas (Suciedad).

In [35]:
#Creo una tabla Espesor vs Tipo de Defecto
tabla = pd.crosstab(df['Rango_Espesor'], df['Tipo_Defecto'], margins=True)

#ORDENO FILAS: Separo la fila 'All', ordeno el resto por la columna 'All' y volvemos a unir
filas_ordenadas = tabla.iloc[:-1].sort_values(by='All', ascending=False)
tabla_final = pd.concat([filas_ordenadas, tabla.iloc[[-1]]])

#ORDENO COLUMNAS: Busco el orden de las columnas usando la fila 'All' (sin incluir el propio 'All')
columnas_por_cantidad = tabla_final.loc['All'].drop('All').sort_values(ascending=False).index

#Reorganizo las columnas de la tabla final dejando 'All' al extremo derecho
tabla_final[list(columnas_por_cantidad) + ['All']]


KeyError: 'Rango_Espesor'

De acuerdo a la tabla anterior parece que:   
* En los espesores más finos es más probable tener rayas superficiales.  
* Para espesores medios es más fácil que se produzcan abolladuras.  
* Y en espesores gruesos parece que la visión artificial no es capaz de identificar la mayoría de los defectos, asignándolos a 'Otro'.

## 4.Analysis

### 4.1 Análisis univariante

In [17]:
# Importamos las nuevas funciones desde tu carpeta utils
from utils.funciones import plot_porcentaje_defectos

# 1. Tu primer gráfico pensado
plot_porcentaje_defectos(df)

ImportError: cannot import name 'plot_porcentaje_defectos' from 'utils.funciones' (c:\Users\marti\Documents\GitHub\2026-10-BILBAO-FT-Data-Science-LauraML\2-Data_Analysis\EDA\Laura_Mtnez_EDA_Steel_Plates_Faults\src\utils\funciones.py)

#### CONCLUSIONES  
* El defecto más frecuente con diferencia es 'Otro' con un 34.7%. Más de una de cada tres defectos detectados en planta cae en esta categoría genérica. El sistema automático de visión artificial de la fábrica no es capaz de clasificar con precisión estas imperfecciones. Dejar un volumen tan alto (casi el 35%) sin identificar genera una ceguera operativa enorme. Aquí hay una OPORTUNIDAD millonaria para optimizar y reentrenar los algoritmos de clasificación de la planta.  
* Si limpiamos el dataset y dejamos fuera los errores no clasificados por el software, descubrimos el verdadero foco del problema: solo tres tipos de defectos (Abolladura, Surco y Raya) representan casi el 80% de todas las pérdidas por calidad de la planta (77.5%). Concentrar los esfuerzos de mantenimiento y calibración en estas tres áreas específicas erradicará de forma inmediata el grueso de las mermas en la producción de acero.  
* Las manchas químicas (3.7%) y la suciedad en el acero (2.8%) son los defectos menos frecuentes, sumando apenas un 6.5% entre ambos. Este es un punto muy positivo para la gestión de la planta. Te demuestra que los sistemas de descalillado hidráulico (limpieza con agua a alta presión) y los controles de pureza química en los slabs iniciales están funcionando de forma excelente. La suciedad no es el problema de esta fábrica.

In [ ]:
from utils.funciones import pareto_defectos
#Análisis de Pareto sin 'Other_Faults'
pareto_defectos(df)

Si limpiamos el dataset y dejamos fuera los errores no clasificados por el software, descubrimos el verdadero foco del problema: solo tres tipos de defectos (Abolladura, Surco y Raya) representan casi el 80% de todas las pérdidas por calidad de la planta (77.5%). Concentrar los esfuerzos de mantenimiento y calibración en estas tres áreas específicas erradicará de forma inmediata el grueso de las mermas en la producción de acero.